# CI failure prediction on GitHub Actions - CART decision tree

Trains and evaluates the from-scratch CART tree (`tree/cart.py`, Breiman et al., 1984) on the **pooled** dataset (the enabled targets of `config/targets.json`, analysed as one CI workflow), with the frozen feature lists from `feature_selection.ipynb`.

**Protocol.** The dev segment (first 20% of the pooled runs) is followed by 5 expanding-window folds. Each train set is dev + the window so far; its last 20% is the **validation** tail. The hyperparameters, the pruning alpha and the decision threshold are chosen on validation only (`tree/tune.py`); each test fold is read once, after the tree is final.

**Per source.** Each fold's test rows are also scored per source: `run_cv` adds one row per fold and source next to the pooled row (`source = "all"`). The model is the same; only the scored rows differ.

**Targets.** T2 (runs whose previous run in the same lineage passed, or that open a lineage) is the main target; T1 (all runs) is secondary, for the comparison with Parrot.

**Baselines.** Parrot (previous outcome of the same lineage; on T2 always "pass"), majority class, and sklearn's `DecisionTreeClassifier` with the same choices per fold (a reference for the implementation, not a competitor).

**Random forest.** Section 9 repeats the CV with a forest of 100 bagged CART trees (`tree/forest.py`, Breiman, 2001): a stronger but not explainable comparison point.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from sklearn.metrics import matthews_corrcoef
from sklearn.tree import DecisionTreeClassifier
import json

from acise_p2_cidt.analysis import features as fs
from acise_p2_cidt.analysis.features import load_features
from acise_p2_cidt.collect.targets import load_targets, slug
from acise_p2_cidt.data.load import load, pool_name, repo_of, subset
from acise_p2_cidt.metrics.metrics import majority, parrot
from acise_p2_cidt.stats import tests as st
from acise_p2_cidt.training.cv import dev_split, folds, prepare, run_cv
from acise_p2_cidt.tree import forest, tune
from acise_p2_cidt.tree.cart import CART
from acise_p2_cidt.validate.live import MODEL_DIR
from acise_p2_cidt.viz import plots

SOURCES = [slug(c.target) for c in load_targets()]
DATASET = pool_name(SOURCES)
TARGETS = ("T2", "T1")
METRICS = ["mcc", "f1", "precision", "recall", "accuracy", "pct_builds_saved", "pct_failures_caught"]
data = {t: load(SOURCES, t) for t in TARGETS}
feats = {t: load_features(t, DATASET) for t in TARGETS}
fold_lists = {t: folds(data[t]) for t in TARGETS}
splits = {t: [prepare(data[t], feats[t], f) for f in fold_lists[t]] for t in TARGETS}
display(
    Markdown(f"**Pooled dataset `{DATASET}`** ({len(SOURCES)} sources): rows, failures, dev rows and frozen features per target"),
    pd.DataFrame(
        {
            t: {
                "rows": ds.meta["n_rows"],
                "failed": ds.meta["n_failed"],
                "fail_%": round(100 * ds.y.mean(), 2),
                "dev_rows": ds.meta["n_dev"],
                "features": len(feats[t]),
            }
            for t, ds in data.items()
        }
    ),
)

## 1. CART walkthrough (T2, dev split)

One tree on the dev split (head of dev = fit rows, tail = validation), depth 3, `class_weight="balanced"`. Each internal node is a test `x <= t` in raw units (no scaling, see the plan); each leaf gives P(fail) as the weighted share of failed runs that reach it.

In [ ]:
dev = dev_split(data["T2"], feats["T2"])
demo = CART(max_depth=3, min_samples_leaf=10, class_weight="balanced").fit(dev.X_fit, dev.y_fit)
print(f"depth {demo.depth_}, leaves {demo.n_leaves_}\n")
print(demo.export_text())

`explain` gives the path of one run: the tests it passed, then the leaf. Shown for the first failed run of the validation tail.

In [ ]:
i = int(np.flatnonzero(dev.y_val.to_numpy())[0])
print("\n".join(demo.explain(dev.X_val.iloc[i])))
print("actual: fail")
plots.save(
    plots.tree_icicle(demo.tree_, demo.feature_names_, max_depth=3).update_layout(title="Demo tree (T2 dev, depth 3)"),
    "tree_demo_T2",
)

**Reference check.** sklearn's tree with the same parameters. When two splits have exactly the same gain, sklearn takes one at random (it shuffles the features), CART takes the first column, so the two trees can differ on ties only. `tests/test_cart.py` checks that every difference is such a tie. sklearn fits on float32, so its input is cast here.

In [ ]:
X32 = dev.X_fit.astype("float32")
sk = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, class_weight="balanced", random_state=0).fit(X32, dev.y_fit)
agree = np.mean(demo.predict(dev.X_fit) == sk.predict(X32).astype(bool))
print(f"root impurity: CART {demo.tree_.impurity[0]:.6f}, sklearn {sk.tree_.impurity[0]:.6f}")
print(f"leaves: CART {demo.n_leaves_}, sklearn {sk.get_n_leaves()}; training predictions agree on {agree:.1%}")

## 2. Overfitting control (T2)

**Depth.** In each CV fold, an unpruned tree per depth (`min_samples_leaf=10`, Gini, balanced), scored at threshold 0.5 on the fit rows and on the validation tail. A train line that keeps rising while the validation line stays flat is overfitting.

In [ ]:
BASE = {"min_samples_leaf": 10, "criterion": "gini", "class_weight": "balanced"}
rows = []
for k, s in enumerate(splits["T2"]):
    for d in tune.GRID["max_depth"]:
        m = CART(max_depth=d, **BASE).fit(s.X_fit, s.y_fit)
        for name, X, y in (("train", s.X_fit, s.y_fit), ("val", s.X_val, s.y_val)):
            rows.append({"fold": k, "max_depth": d, "split": name, "mcc": matthews_corrcoef(y, m.predict(X))})
plots.save(
    plots.validation_curve(pd.DataFrame(rows), "max_depth").update_layout(title="MCC vs max_depth, unpruned (T2, 5 folds)"),
    "tree_valcurve_depth_T2",
)

**Pruning.** A depth-8 tree on the dev split, scored at every alpha of its cost-complexity path. `tune.choose_alpha` takes the alpha with the best validation MCC (ties: the larger alpha, the smaller tree).

In [ ]:
deep = CART(max_depth=8, **BASE).fit(dev.X_fit, dev.y_fit)
rows = []
for a in np.unique(deep.cost_complexity_path()[0]):
    p = deep.pruned(a)
    for name, X, y in (("train", dev.X_fit, dev.y_fit), ("val", dev.X_val, dev.y_val)):
        # alpha 0 (the unpruned tree) is drawn at 1e-6, so it fits on the log axis
        rows.append({"ccp_alpha": max(a, 1e-6), "split": name, "mcc": matthews_corrcoef(y, p.predict(X)), "leaves": p.n_leaves_})
alpha_df = pd.DataFrame(rows)
print("chosen alpha:", tune.choose_alpha(deep, dev))
plots.save(
    plots.validation_curve(alpha_df, "ccp_alpha", log_x=True)
    .update_layout(title="MCC along the pruning path (T2 dev, depth 8)")
    .update_yaxes(title_text="mcc (dev split)"),
    "tree_valcurve_alpha_T2",
)

## 3. Tuning on the dev split

The full grid (`max_depth` x `min_samples_leaf` x criterion x class weight = 84 points) on the dev split, as in project 1. For each point: fit, prune on validation, tune the threshold on validation. The CV below repeats this inside every fold, so this table only shows how flat the grid is.

In [ ]:
best_dev, grid_dev = tune.grid_search(dev)
print(best_dev)
display(
    Markdown("**Grid search on the dev split** (T2): the 10 best grid points by validation MCC"),
    grid_dev.sort_values("val_mcc", ascending=False).head(10).round(4),
)

## 4. Cross-validation

Each fold: grid search on the validation tail, then one prediction of the test fold. Hyperparameters and the pruning alpha are chosen on the whole validation tail. The decision threshold is then tuned **per source**, on each source's validation rows: one pooled threshold follows each source's failure rate, so it would almost never flag a low-rate source and would flag a high-rate source too often. A source with fewer than 10 validation failures keeps the pooled threshold. The source of a run is known before it starts, so this reads no future information.

Methods:

- `cart`: CART with per-source thresholds; `cart_pooled_thr`: the same trees with the single pooled threshold.
- `sklearn`: sklearn's tree with CART's choice of each fold (params, alpha, thresholds); it shows how much the from-scratch implementation itself changes the result.
- `parrot`, `majority`: the history baselines. On T2 Parrot predicts all-pass, since every T2 run's previous run of its lineage passed.
- `source_rate`: knows only the source. P(fail) is the source's failure rate on the fit rows, with one threshold. A model close to it mostly tells the sources apart.

In [ ]:
def sklearn_ref(choices: list[tune.Choice]):
    """sklearn tree with CART's choice per fold; run_cv visits the folds in order."""
    it = iter(choices)

    def fit_predict(split):
        c = next(it)
        m = DecisionTreeClassifier(**c.params, ccp_alpha=c.alpha, random_state=0).fit(split.X_fit, split.y_fit)
        return tune.apply_thresholds(m.predict_proba(split.X_test)[:, 1], split.src_test, c)

    return fit_predict


results, payloads = {}, {}
for t in TARGETS:
    print(f"Target: {t}")
    ds, f = data[t], feats[t]
    cart, payloads[t] = run_cv(ds, f, tune.fit_predict, "cart")
    choices = [payloads[t][k]["choice"] for k in sorted(payloads[t])]
    results[t] = pd.concat(
        [
            cart,
            run_cv(ds, f, sklearn_ref(choices), "sklearn")[0],
            run_cv(ds, f, parrot, "parrot")[0],
            run_cv(ds, f, majority, "majority")[0],
            run_cv(ds, f, tune.source_rate, "source_rate")[0],
        ],
        ignore_index=True,
    )

Choice per fold, with the validation MCC it was chosen on next to the test MCC.

In [ ]:
test_mcc = {t: results[t].query("method == 'cart' and source == 'all'").set_index("fold")["mcc"] for t in TARGETS}
display(
    Markdown("**CART choice per fold**: grid parameters, pruning alpha, pooled and per-source thresholds, tree size, the validation MCC it "
             "was chosen on, and the pooled test MCC"),
    pd.DataFrame(
        [
            {
                "target": t,
                "fold": k,
                **p["choice"].params,
                "alpha": p["choice"].alpha,
                "threshold": p["choice"].threshold,
                "per_source_thr": ", ".join(f"{v:.2f}" for v in (p["choice"].thresholds or {}).values()),
                "depth": p["tree"].depth_,
                "leaves": p["tree"].n_leaves_,
                "val_mcc": p["choice"].val_mcc,
                "test_mcc": test_mcc[t][k],
            }
            for t in TARGETS
            for k, p in payloads[t].items()
        ]
    ).round(4),
)

Mean +/- std over the 5 test folds (sample std, ddof=1), first on the pooled test rows, then per source on each source's own test rows. A source with few failures in a fold gives a noisy score: the last table counts the failures per fold and source. In the box plots (pooled) the dashed line is the mean and the diamond spans mean +/- 1 std. The line chart shows each fold separately: a fold that is hard for every method is a property of that period, not of the model.

In [ ]:
def mean_std(r: pd.DataFrame, metrics: list[str], by: tuple[str, ...] = ("method",)) -> pd.DataFrame:
    """'mean +/- std' over the folds (sample std, ddof=1), per `by` group."""
    g = r.groupby(list(by))[metrics]
    return g.mean().map(lambda v: f"{v:.3f}") + " +/- " + g.std().map(lambda v: f"{v:.3f}")


def pooled_rows(r: pd.DataFrame) -> pd.DataFrame:
    return r[r["source"] == "all"]


for t in TARGETS:
    r = results[t]
    display(
        Markdown(f"**CV results, pooled** ({t}, mean +/- std over the 5 test folds)"),
        mean_std(pooled_rows(r), METRICS),
        Markdown(f"**CV results per source** ({t}, mean +/- std over the 5 test folds)"),
        mean_std(r[r["source"] != "all"], ["mcc", "recall", "precision"], by=("source", "method")),
        Markdown(f"**Failures per test fold and source** ({t})"),
        r.query("method == 'cart'").pivot(index="source", columns="fold", values="n_fail"),
    )
    long = pooled_rows(r).melt(id_vars=["method", "fold"], value_vars=["mcc", "f1", "precision", "recall"], var_name="metric")
    display(plots.save(plots.metrics_box(long).update_layout(title=f"CV metrics per fold, pooled ({t})"), f"tree_cv_box_{t}"))
    display(plots.save(
        px.line(pooled_rows(r), x="fold", y="mcc", color="method", markers=True)
        .update_layout(title=f"Test MCC per fold, pooled ({t})"),
        f"tree_cv_folds_{t}",
    ))

## 5. Statistical tests

Paired two-sided Wilcoxon signed-rank test and the Vargha-Delaney A12 (P(first method > second); 0.5 = no difference) on the test MCC. With only the 5 pooled folds, the smallest reachable p is 0.0625, so no comparison could reach 0.05. The pairs are therefore the **(fold, source) cells**: the MCC of one source's test rows in one fold. Cells with fewer than 10 test failures are left out (`MIN_TEST_FAILURES`). Cells of one fold share a model and a time window, so they are not independent, and the p-values are optimistic. The per-fold pooled table follows as the plain view.

In [ ]:
def compare_rows(r: pd.DataFrame, first: str, others: tuple[str, ...], label: str) -> list[dict]:
    cells = st.per_cell(r, "mcc")
    return [{label: f"{first} vs {o}", "cells": len(cells[[first, o]].dropna()), **st.compare(cells, first, o)}
            for o in others if o in cells]


for t in TARGETS:
    display(
        Markdown(f"**CART vs. the baselines** ({t}): paired Wilcoxon and A12 on the (fold, source) MCC cells"),
        pd.DataFrame(compare_rows(results[t], "cart", ("cart_pooled_thr", "source_rate", "parrot", "majority", "sklearn"), "test")).round(4),
        Markdown(f"**Test MCC per fold and method**, pooled ({t})"),
        st.per_fold(results[t], "mcc").round(3),
    )

## 6. What the tree uses

**Repository proxy.** Each source has its own failure rate and its own scale of loads, churn and durations, so a tree can first recognise the source and then predict its rate. Two checks: `cart` against `source_rate` in section 5 (how much of the pooled MCC knowing the source alone gives), and the per-source MCC of section 4 (a tree that only recognised sources would score about 0 within each source). The top splits of the last fold's tree follow below.

Impurity-decrease importance of each fold's final (pruned) tree. The table gives the mean and std over the folds; the plot shows one point per fold and the dashed line is the mean. No ± std band here: importances are bounded at 0 and skewed (a feature can be large in one fold and 0 in the others), so mean − std would fall below 0.

In [ ]:
for t in TARGETS:
    imp = pd.DataFrame({k: p["tree"].feature_importances_ for k, p in payloads[t].items()}, index=feats[t])
    imp = imp.loc[imp.mean(axis=1).sort_values(ascending=False).index]
    display(
        Markdown(f"**Impurity-decrease importance per fold** ({t}, final tree of each fold), with mean and std"),
        imp.assign(mean=imp.mean(axis=1), std=imp.std(axis=1)).round(3),
    )
    long = imp.rename_axis("feature").reset_index().melt(id_vars="feature", var_name="fold", value_name="importance")
    display(plots.save(
        px.box(long, x="feature", y="importance", points="all", hover_data=["fold"])
        .update_traces(boxmean=True)
        .update_layout(title=f"Impurity-decrease importance per fold ({t})"),
        f"tree_importance_{t}",
    ))

The tree of the last fold (largest train set), top 3 levels. Box width = runs reaching the node, colour = P(fail).

In [ ]:
last = payloads["T2"][max(payloads["T2"])]["tree"]
print(last.export_text(max_depth=3))
plots.save(
    plots.tree_icicle(last.tree_, last.feature_names_, max_depth=3).update_layout(title="CART, last CV fold (T2, top 3 levels)"),
    "tree_icicle_T2",
)

## 7. Errors and the cost trade-off

Confusion matrix of CART, summed over the 5 test folds: pooled, then per source.

In [ ]:
def pooled(t: str, method: str, source: str = "all") -> pd.Series:
    r = results[t]
    return r[(r["method"] == method) & (r["source"] == source)][["tn", "fp", "fn", "tp"]].sum()


for t in TARGETS:
    c = pooled(t, "cart")
    display(plots.save(
        plots.confusion(np.array([[c.tn, c.fp], [c.fn, c.tp]]))
        .update_layout(title=f"CART confusion, 5 test folds summed, all sources ({t})"),
        f"tree_confusion_{t}",
    ))
    per_source = results[t].query("method == 'cart' and source != 'all'").groupby("source")[["tn", "fp", "fn", "tp"]].sum()
    display(Markdown(f"**CART confusion per source** ({t}, summed over the 5 test folds)"), per_source)

**Trade-off.** Each fold's tree scores its test fold; the probabilities are pooled and one threshold is swept over all of them. A higher threshold skips more builds (`% builds saved`) and catches fewer failures. The star is CART with its tuned per-fold thresholds; the diamond is Parrot. The dotted line is a random guess that flags a share q of the builds and so catches a share q of the failures (caught = 100 - saved): the distance above it is what the model adds.

**Pooled, not averaged.** Every point here counts all test runs together, so folds with more failures weigh more, and it differs from the fold means above. A single global threshold read off this curve uses the test labels, so it is a diagnosis of the threshold choice, not a result.

In [ ]:
def point(c: pd.Series) -> tuple[float, float]:
    return 100 * (c.tn + c.fn) / c.sum(), 100 * c.tp / (c.tp + c.fn)


for t in TARGETS:
    proba = np.concatenate([payloads[t][k]["tree"].predict_proba(s.X_test)[:, 1] for k, s in enumerate(splits[t])])
    y = np.concatenate([data[t].y.iloc[f.test].to_numpy() for f in fold_lists[t]]).astype(bool)
    curve = pd.DataFrame(
        [
            {"threshold": th, "saved": 100 * np.mean(proba <= th), "caught": 100 * np.mean(proba[y] > th)}
            for th in np.unique(np.r_[0.0, proba])
        ]
    )
    fig = plots.tradeoff(curve, point(pooled(t, "cart")), "CART (tuned thresholds)")
    fig.add_scatter(x=[0, 100], y=[100, 0], mode="lines", name="random", line={"dash": "dot", "color": "grey"})
    px_ = point(pooled(t, "parrot"))
    fig.add_scatter(x=[px_[0]], y=[px_[1]], mode="markers", name="Parrot", marker={"symbol": "diamond", "size": 12})
    display(plots.save(fig.update_layout(title=f"Failures caught vs builds saved ({t}, 5 test folds pooled)"), f"tree_tradeoff_{t}"))

## 8. E1 and E3

Two experiments next to the pooled model (E2, `cart` above), on T2 and on the **same test rows** as E2: the pooled folds are kept, and only the rows a model is trained on change. So every (fold, source) cell of E1 and E3 pairs with the same cell of E2.

- **E1, one tree per source**: fit and validation rows of one source only, its own feature list (`feature_selection.ipynb`, section 6.1), scored on that source's test rows. Sources without a list are skipped.
- **E3, leave one repository out**: features selected on the pool's dev rows without repository X (same selection parameters as the pooled list), trees fitted on the pool without X, scored on X's test rows. X never appears in validation, so it gets the pooled threshold. To keep the run time bounded, each fold reuses E2's grid choice of that fold; alpha and the threshold are tuned again on the validation rows without X.

CART only: the forest would multiply the run time.

In [ ]:
E_T = "T2"
ds = data[E_T]
src = ds.source.to_numpy()
repo = ds.source.map(repo_of).to_numpy()
sel_meta = json.loads((fs.OUT_DIR / f"selected_features_{E_T}_{DATASET}.json").read_text())["meta"]
e1_feats = {s: load_features(E_T, s) for s in SOURCES if (fs.OUT_DIR / f"selected_features_{E_T}_{s}.json").exists()}
e2_choices = [payloads[E_T][k]["choice"] for k in sorted(payloads[E_T])]


def with_e2_params(choices: list[tune.Choice]):
    """tune.fit_predict with E2's grid choice of each fold; run_cv visits the folds in order."""
    it = iter(choices)
    return lambda split: tune.fit_predict(split, next(it).params)


e_parts = []
for s, f in e1_feats.items():
    print(f"E1: {s}")
    r, _ = run_cv(ds, f, tune.fit_predict, "cart_e1", train=src == s, test=src == s)
    e_parts.append(r[r["source"] == s])
for x in sorted({repo_of(s) for s in e1_feats}):
    print(f"E3: without {x}")
    out = repo != x
    sel_x = fs.select_features(
        subset(ds, out, dataset=f"{DATASET}-without-{x}"), **sel_meta["params"], exclude=tuple(sel_meta["excluded"])
    )
    r, _ = run_cv(ds, sel_x.features, with_e2_params(e2_choices), "cart_e3", train=out, test=~out)
    e_parts.append(r[(r["source"] != "all") & (r["method"] == "cart_e3")])
e_results = pd.concat(e_parts, ignore_index=True)

In [ ]:
ecmp = pd.concat([results[E_T].query("method == 'cart' and source != 'all'"), e_results], ignore_index=True)
display(
    Markdown(f"**E1, E2 and E3 per source** ({E_T}, test MCC mean +/- std over the 5 folds; `cart` = E2, the pooled model)"),
    mean_std(ecmp, ["mcc"], by=("source", "method")).unstack("method"),
)
rows = [
    {"source": s, "comparison": f"{a} vs {b}", **st.compare(st.per_cell(ecmp[ecmp.source == s], "mcc"), a, b)}
    for s in sorted(ecmp.source.unique())
    for a, b in (("cart", "cart_e1"), ("cart_e3", "cart_e1"))
    if {a, b} <= set(ecmp.query("source == @s").method)
]
display(
    Markdown(f"**E2 and E3 against E1, per source** ({E_T}): 5 folds per source, so only the mean difference and A12 "
             "are informative (smallest p = 0.0625); cells with < 10 test failures are left out"),
    pd.DataFrame(rows)[["source", "comparison", "mean_diff", "n", "A12"]].round(4),
    Markdown(f"**E2 and E3 against E1, all sources** ({E_T}): paired Wilcoxon and A12 on the (fold, source) MCC cells"),
    pd.DataFrame(
        compare_rows(ecmp, "cart", ("cart_e1", "cart_e3"), "test") + compare_rows(ecmp, "cart_e3", ("cart_e1",), "test")
    ).round(4),
)

## 9. Random forest (bagging over CART)

A single CART is a weak learner. A random forest (Breiman, 2001; `tree/forest.py`) averages 100 of our own CART trees. Each tree is fitted on a bootstrap sample of the fit rows and draws sqrt(p) columns at every node, so the trees differ and part of their errors cancel. P(fail) is the mean of the trees' leaf shares.

**Protocol.** Same folds, features and validation tail as CART, scored pooled and per source. The trees are not pruned. The grid (`max_depth` in {8, None} x `min_samples_leaf` in {5, 20} x class weight = 8 points) and the decision threshold are chosen on validation MCC; each test fold is read once. The seed is fixed (`random_state=0`), so each fold is run once, as for CART.

**What it costs.** A forest has no single readable path, so it is not the model we explain. It shows how much accuracy the single tree gives up for its explanation.

In [ ]:
rf_payloads = {}
for t in TARGETS:
    print(f"Target: {t}")
    rf, rf_payloads[t] = run_cv(data[t], feats[t], forest.fit_predict, "forest")
    # drop an earlier forest run first, so the cell can be re-run
    results[t] = pd.concat([results[t].query("method != 'forest'"), rf], ignore_index=True)

Choice per fold, with the validation MCC it was chosen on, the test MCC, and CART's test MCC on the same fold.

In [ ]:
rf_mcc = {t: results[t].query("method == 'forest' and source == 'all'").set_index("fold")["mcc"] for t in TARGETS}
display(
    Markdown("**Forest choice per fold**: grid parameters, threshold, mean tree depth, the validation MCC it was "
             "chosen on, the pooled test MCC, and CART's pooled test MCC on the same fold"),
    pd.DataFrame(
        [
            {
                "target": t,
                "fold": k,
                **p["choice"].params,
                "threshold": p["choice"].threshold,
                "mean_depth": np.mean([tr.depth_ for tr in p["forest"].estimators_]),
                "val_mcc": p["choice"].val_mcc,
                "test_mcc": rf_mcc[t][k],
                "cart_test_mcc": test_mcc[t][k],
            }
            for t in TARGETS
            for k, p in rf_payloads[t].items()
        ]
    ).round(4),
)

Mean +/- std over the 5 test folds, pooled and per source, next to CART and the baselines (same tables and plots as section 4).

In [ ]:
COMPARE = ["forest", "cart", "source_rate", "parrot", "majority"]
for t in TARGETS:
    r = results[t][results[t].method.isin(COMPARE)]
    display(
        Markdown(f"**CV results, pooled, forest vs. CART and the baselines** ({t}, mean +/- std over the 5 test folds)"),
        mean_std(pooled_rows(r), METRICS),
        Markdown(f"**CV results per source, forest vs. CART and the baselines** ({t}, mean +/- std over the 5 test folds)"),
        mean_std(r[r["source"] != "all"], ["mcc", "recall", "precision"], by=("source", "method")),
    )
    long = pooled_rows(r).melt(id_vars=["method", "fold"], value_vars=["mcc", "f1", "precision", "recall"], var_name="metric")
    display(plots.save(plots.metrics_box(long).update_layout(title=f"CV metrics per fold, pooled, forest vs CART ({t})"), f"forest_cv_box_{t}"))
    display(plots.save(
        px.line(pooled_rows(r), x="fold", y="mcc", color="method", markers=True)
        .update_layout(title=f"Test MCC per fold, pooled, forest vs CART ({t})"),
        f"forest_cv_folds_{t}",
    ))

**Statistical tests.** As in section 5: paired Wilcoxon and A12 = P(forest > other) on the (fold, source) MCC cells with at least 10 test failures.

In [ ]:
for t in TARGETS:
    display(
        Markdown(f"**Forest vs. CART and the baselines** ({t}): paired Wilcoxon and A12 on the (fold, source) MCC cells"),
        pd.DataFrame(
            compare_rows(results[t], "forest", ("forest_pooled_thr", "cart", "source_rate", "parrot", "majority"), "test")
        ).round(4),
    )

**What the forest uses.** Impurity-decrease importance averaged over the 100 trees, then over the 5 folds, next to CART's mean from section 6. The column draw forces the forest to split on features the single tree never reaches, so its importances are spread more evenly.

In [ ]:
for t in TARGETS:
    imp = pd.DataFrame(
        {
            "forest": np.mean([p["forest"].feature_importances_ for p in rf_payloads[t].values()], axis=0),
            "cart": np.mean([p["tree"].feature_importances_ for p in payloads[t].values()], axis=0),
        },
        index=feats[t],
    ).sort_values("forest", ascending=False)
    display(Markdown(f"**Mean impurity-decrease importance**, forest vs. CART ({t}, mean over the 5 folds)"), imp.round(3))
    long = imp.rename_axis("feature").reset_index().melt(id_vars="feature", var_name="model", value_name="importance")
    display(plots.save(
        px.bar(long, x="feature", y="importance", color="model", barmode="group")
        .update_layout(title=f"Mean impurity-decrease importance, forest vs CART ({t})"),
        f"forest_importance_{t}",
    ))

**Trade-off.** The pooled curve of section 7, for the forest and CART on the same axes. The forest gives almost every run its own P(fail), so its curve is drawn at 201 quantile thresholds. The star is the forest with its tuned per-fold thresholds. As in section 7, the curve uses the test labels and is a diagnosis of the ranking, not a result.

In [ ]:
def sweep(proba: np.ndarray, y: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        [{"threshold": th, "saved": 100 * np.mean(proba <= th), "caught": 100 * np.mean(proba[y] > th)} for th in thresholds]
    )


for t in TARGETS:
    y = np.concatenate([data[t].y.iloc[f.test].to_numpy() for f in fold_lists[t]]).astype(bool)
    p_rf = np.concatenate([rf_payloads[t][k]["forest"].predict_proba(s.X_test)[:, 1] for k, s in enumerate(splits[t])])
    p_cart = np.concatenate([payloads[t][k]["tree"].predict_proba(s.X_test)[:, 1] for k, s in enumerate(splits[t])])
    fig = plots.tradeoff(
        sweep(p_rf, y, np.unique(np.r_[0.0, np.quantile(p_rf, np.linspace(0, 1, 201))])),
        point(pooled(t, "forest")),
        "forest (tuned thresholds)",
    )
    fig.data[0].update(name="forest", showlegend=True)
    c = sweep(p_cart, y, np.unique(np.r_[0.0, p_cart]))
    fig.add_scatter(x=c.saved, y=c.caught, mode="lines+markers", name="CART")
    fig.add_scatter(x=[0, 100], y=[100, 0], mode="lines", name="random", line={"dash": "dot", "color": "grey"})
    px_ = point(pooled(t, "parrot"))
    fig.add_scatter(x=[px_[0]], y=[px_[1]], mode="markers", name="Parrot", marker={"symbol": "diamond", "size": 12})
    display(plots.save(fig.update_layout(title=f"Failures caught vs builds saved, forest vs CART ({t}, 5 test folds pooled)"), f"forest_tradeoff_{t}"))

## 10. Save the results

The CV rows (pooled and per source, every method) and the E1/E3 rows go to `results/model/`, where `validation.ipynb` reads them next to the re-test.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for t in TARGETS:
    print(results[t].to_csv(MODEL_DIR / f"cv_results_{t}_{DATASET}.csv", index=False) or f"cv_results_{t}_{DATASET}.csv")
e_results.to_csv(MODEL_DIR / f"e_results_{E_T}_{DATASET}.csv", index=False)
print(f"e_results_{E_T}_{DATASET}.csv")

## Observations

_To be written after the run on the pooled data._